In [1]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = "../the_execution_node/data/backtest_5m_3yr.parquet"
MODEL_PATH = "../the_models/meta_labeler_v2.json"
UNIVERSE_PATH = "../the_models/curated_universe.json"

# Load Universe
with open(UNIVERSE_PATH, "r") as f:
    baskets = json.load(f).get("baskets", {})

# Load Brain
meta_labeler = xgb.Booster()
meta_labeler.load_model(MODEL_PATH)

# Load Matrix
df = pd.read_parquet(DATA_PATH)
if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)
df = df.sort_index().ffill().dropna()

print(f"[SUCCESS] Lab Ready: {df.shape[0]} rows loaded.")

[SUCCESS] Lab Ready: 315440 rows loaded.


In [2]:
def simulate_strategy(z_thresh=2.0, ai_thresh=0.50, pt_skew=1.0, sl_skew=2.0, leverage=1.0):
    
    portfolio_returns = pd.Series(0.0, index=df.index)
    trades_executed = 0
    
    for spread_name, params in baskets.items():
        weights = params.get('weights', {})
        half_life = params.get('half_life', 1.0)
        allocation = params.get('capital_allocation', 0.0)
        
        if allocation <= 0: continue
            
        spread_val = pd.Series(0.0, index=df.index)
        for ticker, w in weights.items():
            col = f'{ticker}_price' if f'{ticker}_price' in df.columns else ticker
            if col in df.columns:
                spread_val += df[col] * w
                
        if spread_val.eq(0).all(): continue
                
        window = max(int(half_life * 78), 30)
        rolling_mean = spread_val.rolling(window=window).mean()
        rolling_std = spread_val.rolling(window=window).std().replace(0, np.nan)
        z_score = (spread_val - rolling_mean) / rolling_std
        
        # Inject Custom Z-Score Threshold
        base_signals = pd.Series(0, index=df.index)
        base_signals[z_score < -z_thresh] = 1
        base_signals[z_score > z_thresh] = -1
        
        # Core V2 Features
        price_change = spread_val.diff()
        tick_dir = np.sign(price_change).replace(0, np.nan).ffill().fillna(1)
        signed_vol = 100 * tick_dir
        covar = price_change.rolling(30).cov(price_change.shift(1))
        roll_measure = 2 * np.sqrt(np.abs(covar))
        
        features = pd.DataFrame({
            'frac_diff': price_change, 'volatility': rolling_std,
            'signal_strength': z_score, 'order_flow_imbalance': signed_vol.rolling(50).sum(),
            'effective_spread': roll_measure
        }).dropna()[['frac_diff', 'volatility', 'signal_strength', 'order_flow_imbalance', 'effective_spread']]
        
        valid_idx = features.index
        dmatrix = xgb.DMatrix(features)
        win_probs = meta_labeler.predict(dmatrix)
        
        # Inject Custom AI Confidence Threshold
        meta_mask = pd.Series(win_probs > ai_thresh, index=valid_idx)
        final_signals = pd.Series(0, index=df.index)
        final_signals.loc[valid_idx] = base_signals.loc[valid_idx].where(meta_mask, 0)
        
        vol = spread_val.pct_change().ewm(span=100).std().fillna(0) * np.sqrt(78) 
        in_position, entry_price, entry_idx, current_vol = 0, 0.0, 0, 0.0
        
        for i in range(1, len(df)):
            current_price = spread_val.iloc[i]
            prev_price = spread_val.iloc[i-1]
            signal = final_signals.iloc[i]
            
            if in_position == 0:
                if signal != 0:
                    in_position = signal
                    entry_price = current_price
                    entry_idx = i
                    current_vol = vol.iloc[i] if vol.iloc[i] > 0 else 0.005 
                    trades_executed += 1
            
            elif in_position != 0:
                # FIXED MATH: Apply Leverage properly to the allocation
                tick_ret = (current_price - prev_price) * in_position * (allocation * leverage)
                portfolio_returns.iloc[i] += tick_ret
                
                trade_return = (current_price / entry_price - 1) * in_position if entry_price != 0 else 0
                
                # Inject Custom AFML Triple Barriers
                hit_pt = trade_return >= (current_vol * pt_skew)
                hit_sl = trade_return <= -(current_vol * sl_skew)
                hit_time = (i - entry_idx) >= 120 
                
                if hit_pt or hit_sl or hit_time:
                    in_position, entry_price, entry_idx = 0, 0.0, 0

    cum_returns = portfolio_returns.cumsum()
    max_dd = (cum_returns - cum_returns.cummax()).min()
    total_pl = cum_returns.iloc[-1]
    
    return total_pl, max_dd, trades_executed

In [3]:
scenarios = [
    {"name": "Baseline (Current)", "z": 2.0, "ai": 0.50, "pt": 1.0, "sl": 2.0, "lev": 4.0},
    {"name": "Strict AI, High Lev", "z": 2.0, "ai": 0.65, "pt": 1.0, "sl": 2.0, "lev": 20.0},
    {"name": "Positive Skew (Let Winners Run)", "z": 2.0, "ai": 0.55, "pt": 2.0, "sl": 1.0, "lev": 10.0},
    {"name": "Extreme Z-Score Sniping", "z": 2.8, "ai": 0.50, "pt": 1.5, "sl": 1.5, "lev": 15.0},
    {"name": "The Institutional Mix", "z": 2.2, "ai": 0.60, "pt": 1.5, "sl": 1.0, "lev": 20.0}
]

results = []

print("Running AFML Parameter Sweep...\n")
for s in scenarios:
    pl, dd, trades = simulate_strategy(
        z_thresh=s["z"], ai_thresh=s["ai"], 
        pt_skew=s["pt"], sl_skew=s["sl"], leverage=s["lev"]
    )
    
    results.append({
        "Configuration": s["name"],
        "Total P&L": round(pl, 2),
        "Max Drawdown": round(dd, 2),
        "Trades": trades,
        "RoMD": round(abs(pl / dd), 2) if dd != 0 else 0
    })
    print(f"Finished: {s['name']}")

# Display Leaderboard
results_df = pd.DataFrame(results).sort_values(by="Total P&L", ascending=False)
results_df

Running AFML Parameter Sweep...

Finished: Baseline (Current)
Finished: Strict AI, High Lev
Finished: Positive Skew (Let Winners Run)
Finished: Extreme Z-Score Sniping
Finished: The Institutional Mix


,Configuration,Total P&L,Max Drawdown,Trades,RoMD
4,The Institutional Mix,267.28,-1034.71,1779,0.26
0,Baseline (Current),201.16,-316.39,7150,0.64
1,"Strict AI, High Lev",90.23,-14.13,775,6.39
3,Extreme Z-Score Sniping,66.44,-1621.91,5669,0.04
2,Positive Skew (Let Winners Run),-10.31,-728.20,5856,0.01


In [ ]:
import random

num_tests = 50
fine_tune_results = []

print(f"Running Fine-Grained Monte Carlo Search ({num_tests} iterations)...\n")

for i in range(num_tests):
    # Create microscopic variations around the winning parameters
    test_z = round(random.uniform(1.8, 2.4), 2)     # Baseline was 2.0
    test_ai = round(random.uniform(0.60, 0.72), 2)  # Baseline was 0.65
    test_pt = round(random.uniform(0.8, 1.4), 2)    # Baseline was 1.0
    test_sl = round(random.uniform(1.5, 3.0), 2)    # Baseline was 2.0
    test_lev = 20.0 # Lock leverage to cleanly compare pure mathematical edge
    
    pl, dd, trades = simulate_strategy(
        z_thresh=test_z, ai_thresh=test_ai, 
        pt_skew=test_pt, sl_skew=test_sl, leverage=test_lev
    )
    
    # Calculate RoMD securely
    romd = round(abs(pl / dd), 2) if dd < 0 else (999.99 if pl > 0 else 0)
    
    fine_tune_results.append({
        "AI_Conf": test_ai,
        "Z_Score": test_z,
        "PT_Skew": test_pt,
        "SL_Skew": test_sl,
        "Total P&L": round(pl, 2),
        "Max Drawdown": round(dd, 2),
        "Trades": trades,
        "RoMD": romd
    })

# Display the Top 10 Absolute Best Configurations based strictly on RoMD
fine_tune_df = pd.DataFrame(fine_tune_results).sort_values(by="RoMD", ascending=False)
print("=== TOP 10 MONTE CARLO CONFIGURATIONS (RANKED BY RoMD) ===")
fine_tune_df.head(10)